In [2]:
import pandas as pd
import math
import glob
import os
import torch
import random
from torch.nn.utils.rnn import pad_sequence

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [4]:
files = sorted(glob.glob("COMMENTARY_INTL_MATCH/*.csv"))

df = pd.concat(
    [pd.read_csv(f, usecols=['Commentary_short', 'Commentary']) for f in files if os.path.isfile(f)],
    ignore_index=True
)
df

,Commentary,Commentary_short
0,"on the pads to start from Amir, no swing, work...","Mohammad Amir to Renshaw, 1 run"
1,"drifts down leg this time, no swing whatsoever...","Mohammad Amir to Warner, no run"
2,off the outside half and that races away. Bett...,"Mohammad Amir to Warner, FOUR runs"
3,"leg side-ish again, 140 kph. Amir looking for ...","Mohammad Amir to Warner, 1 leg bye"
4,"much better. 139 kph, coming back in on off, p...","Mohammad Amir to Renshaw, no run"
...,...,...
735686,"low full on middle, Baby goes low and scoops i...","Kumar to Sachin Baby, 2 runs"
735687,"rip-roaring yorker shading outside leg, the ba...","Kumar to Sachin Baby, OUT"
735688,"low full toss on middle, there is the war cry ...","Kumar to Iqbal Abdulla, 1 leg bye"
735689,"full and outside off, backs away and carves it...","Kumar to Sachin Baby, 1 run"


In [5]:
data = df.rename(columns={
    'Commentary_short': 'input',
    'Commentary': 'output'
})
data = data[['input','output']]
data

,input,output
0,"Mohammad Amir to Renshaw, 1 run","on the pads to start from Amir, no swing, work..."
1,"Mohammad Amir to Warner, no run","drifts down leg this time, no swing whatsoever..."
2,"Mohammad Amir to Warner, FOUR runs",off the outside half and that races away. Bett...
3,"Mohammad Amir to Warner, 1 leg bye","leg side-ish again, 140 kph. Amir looking for ..."
4,"Mohammad Amir to Renshaw, no run","much better. 139 kph, coming back in on off, p..."
...,...,...
735686,"Kumar to Sachin Baby, 2 runs","low full on middle, Baby goes low and scoops i..."
735687,"Kumar to Sachin Baby, OUT","rip-roaring yorker shading outside leg, the ba..."
735688,"Kumar to Iqbal Abdulla, 1 leg bye","low full toss on middle, there is the war cry ..."
735689,"Kumar to Sachin Baby, 1 run","full and outside off, backs away and carves it..."


In [6]:
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
data = data.dropna(subset=['input', 'output'])
data

,input,output
1,"Dananjaya to Taylor, no run","full on leg stump, pushed towards short midwic..."
2,"Mohammad Abbas to Hope, no run","floats this full outside off, this swung away ..."
3,"Ngidi to Lakmal, no run",blocks a fuller one into the covers
4,"Ahmed to Finch, 2 runs","length ball on the pads, tucks it away towards..."
5,"Rashid Khan to Balbirnie, 1 run","too full to do anything, pushed off the front ..."
...,...,...
735686,"Dananjaya to Mohammad Saifuddin, no run","full on leg, blocked"
735687,"Perera to Patterson, 1 leg bye","full on leg stump, and he goes down on one kne..."
735688,"Harbhajan Singh to Mishra, 1 run",bunts a quicker delivery on off stump down to ...
735689,"Nortje to Bhanuka, no run",defended from a length


In [5]:
# data.to_csv("commentary_checks.csv")

In [7]:
data['text'] = "[BOS] " + data['input'] + " => " + data['output'] + " [EOS]"
text = data['text'].tolist()

In [8]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size = 5000,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

tokenizer.train_from_iterator(text, trainer=trainer)

In [9]:
PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")

def encode(s):
    return tokenizer.encode(s).ids

def decode(ids):
    return tokenizer.decode(ids)

print(encode("[BOS] Virat     Kohli get's his 100! Brilliant century from him. [EOS]"))
print(decode([2, 2014, 2675, 108, 108, 108, 108, 564, 284, 247, 242, 3220, 4, 4781, 4160, 280, 361, 17, 108, 3]))

[2, 2014, 2675, 108, 108, 108, 108, 564, 284, 247, 242, 3220, 4, 4781, 4160, 280, 361, 17, 108, 3]
 Virat     Kohli get's his 100! Brilliant century from him. 


In [10]:
encoded_data = [encode(t) for t in text]
print(encode("=>"))

[130]


In [ ]:
# print(encode("[PAD]"))
# print(encode("[UNK]"))
# print(encode("[BOS]"))
# print(encode("[EOS]"))

[0]
[1]
[2]
[3]


In [11]:
n1 = int(0.8 * len(encoded_data))
n2 = int(0.9 * len(encoded_data))
train_data = encoded_data[:n2]
val_data = encoded_data[n1:n2]
test_data = encoded_data[n2:]

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def get_batch(split, batch_size=32):
    if split == 'train':
        data = train_data
    elif split == 'val':
        data = val_data
    elif split == 'test':
        data = test_data
    else:
        print("Invalid split input. Please try again!")
        return
    
    idx = random.sample(range(len(data)), batch_size)
    
    x = [torch.tensor(data[i][:-1]) for i in idx]
    y = [torch.tensor(data[i][1:]) for i in idx]
    
    x = pad_sequence(x, batch_first=True, padding_value=0)
    y = pad_sequence(y, batch_first=True, padding_value=0)

    x = x.to(device)
    y = y.to(device)
    
    return x, y

Using device: cuda


In [ ]:
# check = get_batch('train')

In [ ]:
# ix = 10
# print(decode(check[0][ix].tolist())) #decodes will show the same for [0][ix] and [1][ix] because decode ignores BOS and EOS
# print(check[0][ix])
# print(decode(check[1][ix].tolist()))
# print(check[1][ix])

 Morris to Rahane, 1 run => full ball, and he makes room but can only get this down towards long off 
tensor([   2, 1360,  115,  829,   15,  174,  131,  130,  222,  204,   15,  138,
         203, 1001,  726,  236,  545, 1024,  284,  238,  252,  331,  333,  150,
         108,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
       device='cuda:0')
 Morris to Rahane, 1 run => full ball, and he makes room but can only get this down towards long off 
tensor([1360,  115,  829,   15,  174,  131,  130,  222,  204,   15,  138,  203,
        1001,  726,  236,  545, 1024,  284,  238,  252,  331,  333,  150,  108,
           3,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,  

In [13]:
print("Max token id:", max(max(seq) for seq in encoded_data))
print("Vocab size:", tokenizer.get_vocab_size())
print("Max number of tokens in one line:", max(len(seq) for seq in encoded_data))

Max token id: 4999
Vocab size: 5000
Max number of tokens in one line: 308


In [14]:
class position_encoding(torch.nn.Module):

    def __init__(self,emb_dim,max_len=308):
        super().__init__()
        pe = torch.zeros(max_len,emb_dim)
        for j in range(max_len):
            for i in range(emb_dim):
                if i%2 == 0:
                    pe[j][i] = math.sin(j/(10000**(2*(i//2)/emb_dim)))
                else:
                    pe[j][i] = math.cos(j/(10000**(2*(i//2)/emb_dim)))
        self.register_buffer("pe",pe)
    
    def forward(self, input_tensor):
        #input_tensor is B,T,E
        T = input_tensor.shape[1] #changes for every batch cuz of padding
        return input_tensor + 0.05*self.pe[:T].unsqueeze(0)

In [15]:
class self_attention(torch.nn.Module):

    def __init__(self, emb_dim, heads):
        super().__init__()
        self.heads = heads
        self.emb_dim = emb_dim
        self.head_dim = int(self.emb_dim/self.heads)
        
        self.q_proj = torch.nn.Linear(emb_dim, emb_dim)
        self.v_proj = torch.nn.Linear(emb_dim, emb_dim)
        self.k_proj = torch.nn.Linear(emb_dim, emb_dim)

        self.out_proj = torch.nn.Linear(emb_dim, emb_dim)

        self.dropout = torch.nn.Dropout(0.1)

    def forward(self,x):
        b,t,e = x.shape

        q = self.q_proj(x) #b,t,e
        v = self.v_proj(x)
        k = self.k_proj(x)

        q = q.view(b,t,self.heads,self.head_dim).transpose(1,2) #b,h,t,d (h*d=e)
        v = v.view(b,t,self.heads,self.head_dim).transpose(1,2)
        k = k.view(b,t,self.heads,self.head_dim).transpose(1,2)
        
        QK = q@k.transpose(3,2)
        mask = torch.tril(torch.ones_like(QK))
        QK = QK.masked_fill(mask==0,float('-inf'))
        QK = QK/(self.head_dim**0.5)
        # QK = QK - QK.max(3, keepdim=True).values
        # QK = QK.exp()
        # QK = QK/QK.sum(3,keepdim=True)
        QK = torch.softmax(QK, dim=3)
        QK = self.dropout(QK)
        y_faux = QK@v #b,h,t,d
        out = y_faux.transpose(1,2) #b,t,h,d
        out = out.contiguous().view(b,t,e)
        out = self.out_proj(out)
        
        return out

In [16]:
class feed_forward_network(torch.nn.Module):

    def __init__(self, emb_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(emb_dim, 4*emb_dim)
        self.fc2 = torch.nn.Linear(4*emb_dim, emb_dim)
        self.dropout = torch.nn.Dropout(0.1)

    def forward(self,x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [17]:
class block(torch.nn.Module):

    def __init__(self, emb_dim, heads):
        super().__init__()
        self.attn = self_attention(emb_dim,heads)
        self.ln1 = torch.nn.LayerNorm(emb_dim)
        self.ffn = feed_forward_network(emb_dim)
        self.ln2 = torch.nn.LayerNorm(emb_dim)
        self.resid_dropout = torch.nn.Dropout(0.1)
    
    def forward(self,x):
        x = x + self.resid_dropout(self.attn(self.ln1(x)))
        x = x + self.resid_dropout(self.ffn(self.ln2(x)))
        # x = x + self.attn(self.ln1(x))
        # x = x + self.ffn(self.ln2(x))
        return x

In [18]:
class Transformer(torch.nn.Module):

    def __init__(self,vocab_size,emb_dim,heads,layers):
        super().__init__()
        self.C = torch.nn.Embedding(vocab_size,emb_dim)
        self.pe = position_encoding(emb_dim)
        self.blocks = torch.nn.ModuleList([
            block(emb_dim,heads) for _ in range(layers)
        ])
        self.ln_f = torch.nn.LayerNorm(emb_dim)
        self.head = torch.nn.Linear(emb_dim,vocab_size) #this is like W_vocab
        # self.head.weight = self.C.weight
        self.dropout = torch.nn.Dropout(0.1)

    def forward(self,x):
        x = self.C(x)
        x = self.dropout(x)
        x = self.pe.forward(x)

        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)
        logits = self.head(x)

        return logits

In [22]:
emb_dim = 512
vocab_size = 5000
heads = 8
head_dim = int(emb_dim/heads)
layers = 6

model = Transformer(vocab_size,emb_dim,heads,layers).to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10000)

In [24]:
for step in range(5000):
    X,Y = get_batch("train",64)
    logits = model(X)
    B,T,V = logits.shape
    loss = torch.nn.functional.cross_entropy(logits.view(B*T, V), Y.view(B*T),ignore_index=PAD_ID)

    optimizer.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if step%100 == 0:
        print(step,loss.item())
print(step,loss.item())

0 8.628220558166504
100 3.6059839725494385
200 3.485024929046631
300 3.0256571769714355
400 3.3988149166107178
500 2.9847402572631836
600 2.911681890487671
700 2.986865520477295
800 3.0573461055755615
900 2.6221327781677246
1000 2.658384323120117
1100 2.8117284774780273
1200 2.6540122032165527
1300 2.8648219108581543
1400 2.7359354496002197
1500 2.8531551361083984
1600 2.470411539077759
1700 2.721045732498169
1800 2.8536157608032227
1900 2.6347317695617676
2000 2.6248223781585693
2100 2.4383275508880615
2200 2.4159905910491943
2300 2.5340869426727295
2400 2.5950725078582764
2500 2.5469775199890137
2600 2.595555543899536
2700 2.6813435554504395
2800 2.4537711143493652
2900 2.383674144744873
3000 2.5452263355255127
3100 2.3477494716644287
3200 2.34491229057312
3300 2.528290033340454
3400 2.2402162551879883
3500 2.3675220012664795
3600 2.326979160308838
3700 2.394605875015259
3800 2.330315589904785
3900 2.3770689964294434
4000 2.5550832748413086
4100 2.578716516494751
4200 2.4774713516235

In [20]:
-torch.log(torch.tensor(1/5000))

tensor(8.5172)

In [25]:
X,Y = get_batch("val", 128)
logits = model(X)
B,T,V = logits.shape
loss = torch.nn.functional.cross_entropy(logits.view(B*T, V), Y.view(B*T),ignore_index=PAD_ID)
print(loss.item())

#after just self attention it is around 20
#after adding positon encoding it was around 18
#after ffn and before layernorm it is 5.95
#after layernorm with 3000 iters it is 4.35


2.4597928524017334


In [32]:
@torch.no_grad()
def generate_text(input, max_new_tokens=308, temperature=0.8, top_p=0.9):
    model.eval()
    tokens = encode(input)
    x = torch.tensor(tokens,device=device).unsqueeze(0) #(1,T)
    for _ in range(max_new_tokens):
        x_cond = x[:, -308:]
        logits = model(x_cond) #1,T,vocab_size
        last_logits = logits[0,-1,:] #(vocab_size)
        logits = last_logits/temperature
        probs = torch.softmax(logits,dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum_probs = torch.cumsum(sorted_probs,dim=-1)
        mask = cum_probs > top_p
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = False
        sorted_probs[mask] = 0.0
        sorted_probs = sorted_probs / sorted_probs.sum()
        next_token = sorted_idx[torch.multinomial(sorted_probs,num_samples=1)].item()
        x = torch.cat([x,torch.tensor([[next_token]],device=device)],dim=1)
        if next_token == encode("[EOS]")[0]:
            break
    return decode(x[0].tolist())

In [33]:
for i in range(10):
    print(generate_text("[BOS] Kumar to Warner, 3 runs =>"))

 Kumar to Warner, 3 runs => gets nicely timed, but the cover fielder there is a man there, third man is in two men in the deep 
 Kumar to Warner, 3 runs => length outside off, flicked through midwicket, and that's a freebie. And it's a great over from Billings. Warner picks up two 
 Kumar to Warner, 3 runs => good length on off stump, he defends on the front foot through cover and they have to run. Into the deep 
 Kumar to Warner, 3 runs => too full and on the stumps, Warner goes for the drive and skews it over point for a couple 
 Kumar to Warner, 3 runs => a crunching drive through cover. Too easy for Warner. Superb batting 
 Kumar to Warner, 3 runs => back of a length, angling into middle and leg, worked away to the right of midwicket for three. Nicely played, but that's just a single. 
 Kumar to Warner, 3 runs => back of a length on middle stump, Warner tucks it behind square and they get three 
 Kumar to Warner, 3 runs => excellent shot. Warner pokes at this length ball outside of

In [34]:
for i in range(10):
    print(generate_text("[BOS] Cummins to Latham, FOUR runs =>"))

 Cummins to Latham, FOUR runs => shot! Fuller and angling into the pads, he plays inside the line and flicks this fine and he's timed it just landed it a boundary 
 Cummins to Latham, FOUR runs => short, and he pulls hard, gets a thick inside edge that goes wide of short fine leg. Easy 
 Cummins to Latham, FOUR runs => full and angling in, Tharanga flicks this off his toes through midwicket. Nobody at that area 
 Cummins to Latham, FOUR runs => this is a nice length ball, Latham has to move back and pummels it through point for four. 
 Cummins to Latham, FOUR runs => <strong>short and wide outside off, he stretches out and drives hard between cover and mid off!</strong> 
 Cummins to Latham, FOUR runs => too full, too straight, but the same shot is not much timing, enough to get this through the vacant mid-on for four. 
 Cummins to Latham, FOUR runs => 142kph, full and straight, he <strong>drives this neatly through mid off!</strong> 
 Cummins to Latham, FOUR runs => full and wide outsi

In [36]:
for i in range(10):
    print(generate_text("[BOS] Bennett to Sharma, OUT =>"))

 Bennett to Sharma, OUT => <b>caught at point</b> and he's done it. Length ball outside off, and Rohit looks to go over mid-off. But he can't get to the pitch of the ball and the ball goes straight back, and Krunal can't stop it. What a shot from the crowd. 
 Bennett to Sharma, OUT => <strong>caught at long off!</strong> That's the breakthrough of the Powerplay to play the knuckle ball. He was stuck on the crease, Rohit looks to play a length ball in the air, but he didn't quite get hold of it. But he didn't have much feet moving to the pitch and puffs the catch. That's the DRSstruction. 
 Bennett to Sharma, OUT => Another slower delivery outside off, Rohit looks to launch through the covers, and he's taken a simple catch at short mid-off. Doesn't turn to play that. Tries to go over the infield but the ball stays low and ends up slicing it straight to Rohit at point. Sixthird maiden Test match. 
 Bennett to Sharma, OUT => <strong>caught at deep backward square</strong>. What a start fr

In [37]:
for i in range(10):
    print(generate_text("[BOS] Bumrah to Stokes, 1 run =>"))

 Bumrah to Stokes, 1 run => works this through square leg 
 Bumrah to Stokes, 1 run => slower ball, back of a length at middle stump.8ks, flicked to deep midwicket 
 Bumrah to Stokes, 1 run => back of a length outside off, this one's pushed to point with a slightly open face 
 Bumrah to Stokes, 1 run => pushed into the off side, that's a free hit from Stokes, and a comfortable single there's protection in the deep 
 Bumrah to Stokes, 1 run => full and wide, drilled hard to deep extra cover 
 Bumrah to Stokes, 1 run => length outside off, dabbed to third man 
 Bumrah to Stokes, 1 run => slower ball on off stump, and he goes back to flick to deep midwicket 
 Bumrah to Stokes, 1 run => back of a length on off, and worked off his hips to deep square leg 
 Bumrah to Stokes, 1 run => full on middle, whipped to long-on 
 Bumrah to Stokes, 1 run => shuffles across and tucks this with the angle to fine leg 


In [38]:
for i in range(10):
    print(generate_text("[BOS] Mustafizur to Brathwaite, SIX runs =>"))

 Mustafizur to Brathwaite, SIX runs => fuller outside off, and he comes down the track and launches this over long-on for six! 
 Mustafizur to Brathwaite, SIX runs => slower ball on off, he swings hard and connects well over deep midwicket. Not timed the all along the way. It carries all the way 
 Mustafizur to Brathwaite, SIX runs => a half-volley on middle and leg. He's hit it all the way over long-on for a six 
 Mustafizur to Brathwaite, SIX runs => <B>short and slammed over midwicket!</b> He has taken his power! A rocketed slog. This is a half-volley outside off, but he's gone for that he couldn't quite get under the ball 
 Mustafizur to Brathwaite, SIX runs => <b>flicks it over the long-on boundary!</b> Just a short one, but it is still short, and he has no chance to go deep midwicket! 
 Mustafizur to Brathwaite, SIX runs => this is that! Gets away with a short ball outside off and holds his bat out of the way and sends it over deep backward point. That was a half-volley from Cham

In [39]:
for i in range(10):
    print(generate_text("[BOS] Ali to du Plesis, no run =>"))

 Ali to du Plesis, no run => on the stumps, defended back 
 Ali to du Plesis, no run => blocked on off stump 
 Ali to du Plesis, no run => pushed through, back of a length, defended 
 Ali to du Plesis, no run => tossed up and tapped down into the covers 
 Ali to du Plesis, no run => full and straight, defended back 
 Ali to du Plesis, no run => goes back and across as he defends 
 Ali to du Plesis, no run => flicked away to leg 
 Ali to du Plesis, no run => pushed through outside off, jabbed to the off side 
 Ali to du Plesis, no run => googly, and this is defended back to the bowler 
 Ali to du Plesis, no run => too full outside off, and a big stride forward to block 


In [40]:
for i in range(10):
    print(generate_text("[BOS] Santner to Samuels, 2 runs =>"))

 Santner to Samuels, 2 runs => short on middle, back and pulled through midwicket for a couple 
 Santner to Samuels, 2 runs => tossed up, wide of off, he leans out and drives it through extra cover for a couple 
 Santner to Samuels, 2 runs => slower through the air, tossed up outside off, driven hard to the left of sweeper cover. Beterly enough for a couple 
 Santner to Samuels, 2 runs => tossed up and he drives hard, it comes through the covers, but they get a couple 
 Santner to Samuels, 2 runs => low full toss on leg stump, he gets a big stride across and clips it past short fine. Man in the deep makes no one of the deep 
 Santner to Samuels, 2 runs => back of a length outside off, punched down to long-on 
 Santner to Samuels, 2 runs => quicker one on middle, he gets forward and clips it through midwicket 
 Santner to Samuels, 2 runs => short outside off, chopped to sweeper's right. Easy two. 
 Santner to Samuels, 2 runs => makes room and lofts this over cover. Good stop by McCullum

In [41]:
for i in range(10):
    print(generate_text("[BOS] Pandya to Ramdin, 5 wides =>"))

 Pandya to Ramdin, 5 wides => full and angling into middle, it's too wide. A back-of-the-hand slower ball, but Ingram misses the flick. Not given that. Nope. The umpire says not out. The ball-field call from Pakistan. That was very slow and the only question was that ball hit the stumps 
 Pandya to Ramdin, 5 wides => slower ball, full and down leg. Tries to flick it away but misses and it rolls away to the keeper. Appeal for a leg byes 
 Pandya to Ramdin, 5 wides => slower ball on middle, but the ball is going down leg side. Takes the pad on the way to the keeper's right. Not much height, but the umpire isn't interested. 
 Pandya to Ramdin, 5 wides => <b>in the air but safe!</b> Full on middle stump, and the batsman misses his sweep. But it's too high for a wide. Ball tracking down leg 
 Pandya to Ramdin, 5 wides => short of length outside off, goes for the pull but misses and it's the ball goes over him. It's wide down the leg side 
 Pandya to Ramdin, 5 wides => full and wide outside 

In [42]:
for i in range(10):
    print(generate_text("[BOS] Pandya to Ramdin, 1 wide =>"))

 Pandya to Ramdin, 1 wide => banged in short and rising down leg side, and the umpire signals to the leg side, and the umpire thinks about his head and the ball misses off the pads 
 Pandya to Ramdin, 1 wide => short and wide outside off, wildly so he misses his cut. Wide called. 
 Pandya to Ramdin, 1 wide => bouncer at middle stump, ducks under it. Too wide 
 Pandya to Ramdin, 1 wide => full, angled down leg. Looks to flick, but misses. Hits the pad and goes just outside the leg 
 Pandya to Ramdin, 1 wide => slower ball, back of a length outside off. Tries to pull, but mistimes to the left of short midwicket. The ball skews off the bat 
 Pandya to Ramdin, 1 wide => slower bouncer outside off, DK clears the hook and flies over the keeper's head 
 Pandya to Ramdin, 1 wide => good length and in at off stump. Shuffles across and taps it off his pads to the leg side 
 Pandya to Ramdin, 1 wide => short and wide outside off, too much bounce. Narine tries to drag it away, but the ball takes t